# NexPlay · exploración de los datos (00)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fernandoaxelramirezgomez-coder/nexplay/blob/master/notebooks/00_exploracion.ipynb)
[![Ver en nbviewer](https://img.shields.io/badge/ver%20en-nbviewer-orange)](https://nbviewer.org/github/fernandoaxelramirezgomez-coder/nexplay/blob/master/notebooks/00_exploracion.ipynb)

**Objetivo.** Revisar los datos antes de construir el modelo de riesgo: si llegaron completos y sin
contradicciones, qué se limpia o se marca para los modelos de texto y qué decisiones del modelo sostienen.

**Entradas.** El código se clona del tag `codigo-v8` de este repositorio (importa `backend/analisis/` y
`backend/despliegue/utilidades.py`). Los datos se bajan de releases con tag fijo y cada asset se verifica con
su sha256 antes de abrirlo:

| Release | Asset | sha256 | Para qué |
|---|---|---|---|
| `data-v1` | `nexplay_reproducible.db.xz` | `2ef8ef40330385af4c03cd072dccb20fc9a4b635e3929e513235c191d14e9ee7` | 83 juegos y 123,972 reseñas: todas las decisiones |
| `data-v2` | `nexplay_reproducible.db.xz` | `9d5a54f6cbb5f361e397eb043989e592cbff1d553c57ef8a41aae41e2c763d72` | 123 juegos: solo la comparación de la sección 3.6 |
| `data-v1` | `nexplay_extracto.parquet` | `645d685df884c9a352e4306e899c62768ea493375d4b0378c201c7fc6d2d2d6e` | el extracto que lee el 01, para comprobar que los dos usan la misma partición |

La sección 3.9 baja además el modelo de embeddings `sentence-transformers/all-MiniLM-L6-v2`, revisión
`1110a243fdf4706b3f48f1d95db1a4f5529b4d41` (unos 90 MB).

**Salidas.** Tablas y gráficas dentro del notebook. En la carpeta de trabajo quedan el clon (`repo_nexplay/`),
las bases bajadas (`datos_eda/`) y el conjunto limpio de data-v1 (`extracto_eda/limpio_data-v1.parquet`); el modelo de
embeddings queda en la caché de Hugging Face. No escribe nada en el repositorio.

**Cómo correrlo.** En Colab, con el botón de arriba y *Entorno de ejecución → Ejecutar todas*. En local, desde
la raíz del repositorio, `make notebooks` ejecuta los tres notebooks con el código de tu checkout en lugar del tag.
Para saltar los embeddings, pon `CORRER_EMBEDDINGS = False` en la primera celda de código.

**Tiempo estimado.** Unos 5 minutos en local (282 s medidos en un i7 de 8 núcleos, con el modelo de
embeddings ya en caché; la primera vez baja además unos 90 MB). Con las versiones de Colab, 192 s (medido
con las mismas versiones de Colab, no en Colab).

Con `codigo-v8` falta correrlo en Colab. Corrió en Colab el 2026-10-02 con `codigo-v6`, sin errores ni avisos; evidencia en `docs/evidencia/colab/`, junto
con la corrida anterior (2026-09-30, `codigo-v3`). Las salidas guardadas aquí son de una corrida local, para que las 18 gráficas se vean también como PNG en GitHub.

El modelo de riesgo está en `01_modelo_riesgo.ipynb`. Este notebook va en tres bloques:

1. **Validación:** ¿los datos llegaron completos, con el esquema esperado y sin contradicciones?
2. **Limpieza:** qué se quita o se marca para los modelos de texto, con una bitácora.
3. **Exploración:** qué dicen los datos y qué decisiones sostienen.

Cómo leerlo:

- Las decisiones se toman solo con **data-v1** (83 juegos, el corte con el que se entrena el modelo). Los 40
  juegos que llegaron después aparecen solo en la sección 3.6, para comparar.
- Las cifras las calcula cada celda. El texto describe lo que se ve, y un `assert` en la celda de arriba
  falla si los datos dejan de sostenerlo.
- La señal (`Y = 1`) es una reseña negativa escrita antes de 120 minutos de juego, la ventana de reembolso de
  Steam. Es una **señal de arrepentimiento temprano**: no confirma que alguien se arrepintiera.
- Las cuentas viven en `backend/analisis/exploracion.py` y `backend/analisis/limpieza.py`. Este notebook las
  importa; no las copia.

## 0. Preparación

In [1]:
import sys

# True en la versión que se sube al repo: cada gráfica se guarda también como PNG para que se vea en GitHub
# (pide kaleido y un Chrome, ver backend/requirements-dev.txt). En Colab, o sin Chrome, se apaga sola con un
# aviso al configurar las gráficas: ahí se ven interactivas y no hace falta instalar nada más.
EXPORTAR_ESTATICO = True
# Sección 3.9: baja el modelo de embeddings (~90 MB). False la salta.
CORRER_EMBEDDINGS = True

In [2]:
# Colab ya trae pandas, numpy, scikit-learn y plotly: no se reinstalan, porque cambiar sus versiones rompe
# sus binarios. Solo se instala lo que falte. lingua va con versión fija: de ella depende la firma del
# conjunto limpio.
import importlib.util
import subprocess
import sys

requeridos = {"plotly": "plotly", "lingua": "lingua-language-detector==2.2.0"}
if CORRER_EMBEDDINGS:
    requeridos["sentence_transformers"] = "sentence-transformers"
faltantes = [paquete for modulo, paquete in requeridos.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes], check=True)

In [3]:
# Código y datos con referencias fijas: un tag de código y un tag por release, cada asset con su sha256.
GITHUB_REPO = "fernandoaxelramirezgomez-coder/nexplay"
CODIGO_REF = "codigo-v8"

RELEASES = {
    "data-v1": "2ef8ef40330385af4c03cd072dccb20fc9a4b635e3929e513235c191d14e9ee7",  # 83 juegos: todas las decisiones
    "data-v2": "9d5a54f6cbb5f361e397eb043989e592cbff1d553c57ef8a41aae41e2c763d72",  # 123 juegos: solo la sección 3.6
}
# El extracto sin texto que lee 01: sirve para comprobar que los dos notebooks usan la misma partición.
PARQUET_V1_SHA256 = "645d685df884c9a352e4306e899c62768ea493375d4b0378c201c7fc6d2d2d6e"

MODELO_EMBEDDINGS = "sentence-transformers/all-MiniLM-L6-v2"
REVISION_EMBEDDINGS = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"

In [4]:
import os
import shutil

# Callado si sale bien: git avisa que el tag anotado «no es un commit» y que queda en «detached HEAD», y ninguna de
# las dos cosas es un error. Si el clon falla, se ve el mensaje de git.
# Un repo_nexplay que ya estaba se usa solo si es CODIGO_REF: un entorno de Colab que corrió otra versión conserva
# su clon, y con él faltarían los módulos nuevos. Sin .git es la copia que deja make notebooks y se usa tal cual.
if os.path.isdir("repo_nexplay/.git"):
    actual = subprocess.run(["git", "-C", "repo_nexplay", "describe", "--tags", "--exact-match"],
                            capture_output=True, text=True).stdout.strip()
    if actual != CODIGO_REF:
        print(f"repo_nexplay era {actual or 'otro código'}: se borra y se clona {CODIGO_REF}.")
        shutil.rmtree("repo_nexplay")
if os.path.isdir("repo_nexplay"):
    print("repo_nexplay ya existe, no se vuelve a clonar.")
else:
    clon = subprocess.run(
        ["git", "-c", "advice.detachedHead=false", "clone", "--quiet", "--branch", CODIGO_REF, "--depth", "1",
         f"https://github.com/{GITHUB_REPO}.git", "repo_nexplay"],
        capture_output=True, text=True,
    )
    if clon.returncode != 0:
        raise RuntimeError(f"no se pudo clonar {CODIGO_REF}: {clon.stderr.strip()}")

In [5]:
sys.path[:0] = ["repo_nexplay/backend", "repo_nexplay/backend/analisis", "repo_nexplay/backend/modelado"]

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from scipy.optimize import OptimizeWarning

import exploracion as ex  # noqa: E402 (tras sys.path, a propósito)
import limpieza as li  # noqa: E402
import motivos  # noqa: E402
from despliegue.utilidades import descargar_verificado  # noqa: E402

EXPORTAR_ESTATICO = ex.configurar_graficas(EXPORTAR_ESTATICO)
pd.set_option("display.max_colwidth", 230)
pd.set_option("display.width", 200)

# Aviso de scikit-learn 1.6.1 al pasar "iprint" a scipy (L-BFGS-B); no viene de este código ni cambia el resultado.
warnings.filterwarnings("ignore", message="Unknown solver options: iprint", category=OptimizeWarning)

Exportación estática apagada: en Colab no hace falta. Las gráficas se ven interactivas, sin PNG.


In [6]:
DATOS = Path("datos_eda")
rutas = {ref: descargar_verificado(ref, sha, DATOS / f"nexplay_{ref}.db") for ref, sha in RELEASES.items()}
ruta_parquet_v1 = descargar_verificado(
    "data-v1", PARQUET_V1_SHA256, DATOS / "nexplay_extracto_data-v1.parquet", asset="nexplay_extracto.parquet"
)

descargando https://github.com/fernandoaxelramirezgomez-coder/nexplay/releases/download/data-v1/nexplay_reproducible.db.xz ...
descarga verificada (data-v1): 10.9 MB, sha256 OK
datos_eda/nexplay_data-v1.db reconstruida (41.3 MB)
descargando https://github.com/fernandoaxelramirezgomez-coder/nexplay/releases/download/data-v2/nexplay_reproducible.db.xz ...
descarga verificada (data-v2): 15.9 MB, sha256 OK
datos_eda/nexplay_data-v2.db reconstruida (60.4 MB)
descargando https://github.com/fernandoaxelramirezgomez-coder/nexplay/releases/download/data-v1/nexplay_extracto.parquet ...
descarga verificada (data-v1): 1.3 MB, sha256 OK
datos_eda/nexplay_extracto_data-v1.parquet reconstruida (1.3 MB)


In [7]:
# La única carga de datos del notebook. Todo lo demás parte de estas tablas.
juegos_v1, resenas_v1 = ex.cargar_release(rutas["data-v1"])
juegos_v2, resenas_v2 = ex.cargar_release(rutas["data-v2"])
parquet_v1 = pd.read_parquet(ruta_parquet_v1)

# Los 40 externos: están en data-v2 y no en data-v1. Solo se usan en la sección 3.6.
appids_externos = set(juegos_v2["appid"]) - set(juegos_v1["appid"])
assert not appids_externos & set(juegos_v1["appid"]), "un juego externo aparece en data-v1"
juegos_ext = juegos_v2[juegos_v2["appid"].isin(appids_externos)]
resenas_ext = resenas_v2[resenas_v2["appid"].isin(appids_externos)]

# Cada reseña con las columnas de su juego, como las lee entrenar_baseline.py.
df_v1 = ex.con_juego(resenas_v1, juegos_v1)
print(f"data-v1: {len(juegos_v1)} juegos, {len(resenas_v1):,} reseñas")
print(f"externos: {len(juegos_ext)} juegos, {len(resenas_ext):,} reseñas")

data-v1: 83 juegos, 123,972 reseñas
externos: 40 juegos, 60,395 reseñas


# Bloque 1 · Validación

Antes de tocar nada: ¿los datos llegaron completos, con el esquema esperado y sin contradicciones? Cada
chequeo es una función que devuelve una tabla; aquí se muestra la tabla y se hace el `assert`.

## 1.1 Integridad del release

Cada asset se bajó con su sha256 verificado antes de abrirlo: si no coincidiera, la celda de descarga habría
fallado. Aquí se revisan las tablas y los conteos de cada release.

In [8]:
integridad = pd.DataFrame(
    [
        {"release": ref, "sha256": RELEASES[ref][:12] + "…", "tablas": ", ".join(ex.tablas_de(rutas[ref])),
         "juegos": len(juegos), "reseñas": len(resenas)}
        for ref, (juegos, resenas) in {"data-v1": (juegos_v1, resenas_v1), "data-v2": (juegos_v2, resenas_v2)}.items()
    ]
).set_index("release")
display(integridad)

for ref in RELEASES:
    assert {"juegos", "resenas"} <= set(ex.tablas_de(rutas[ref])), f"a {ref} le falta una tabla"
# El corte de entrenamiento está fijo desde que se entrenó el modelo.
assert integridad.loc["data-v1", ["juegos", "reseñas"]].tolist() == [83, 123_972]

,sha256,tablas,juegos,reseñas
release,,,,
data-v1,2ef8ef403303…,"juegos, resenas",83,123972
data-v2,9d5a54f6cbb5…,"juegos, resenas",123,184367


In [9]:
# data-v2 debe traer las reseñas de data-v1 sin cambios: si no, los 83 de la comparación no serían los mismos.
columnas_comparadas = ["appid", "texto", "voted_up", "playtime_at_review", "timestamp_created"]
presentes, celdas_distintas = ex.filas_distintas(resenas_v1, resenas_v2, "recommendationid", columnas_comparadas)
print(f"reseñas de data-v1 presentes en data-v2: {presentes:,} de {len(resenas_v1):,}; celdas distintas: {celdas_distintas}")
assert presentes == len(resenas_v1) and celdas_distintas == 0

reseñas de data-v1 presentes en data-v2: 123,972 de 123,972; celdas distintas: 0


**Resultado.** Los dos releases traen las tablas `juegos` y `resenas`, y data-v1 tiene el corte esperado.
data-v2 contiene data-v1 completo y sin cambios, así que los 40 externos son solo lo que se agregó después.

## 1.2 Esquema: tipos, nulos y qué es cada columna

La última columna sale de `backend/analisis/diccionario.py`, el mismo diccionario que usa la tabla de variables del 01.

In [10]:
esquema_resenas = ex.esquema(rutas["data-v1"], "resenas", resenas_v1)
display(esquema_resenas)
assert (esquema_resenas[ex.QUE_ES] != "").all(), "una columna de resenas no está en backend/analisis/diccionario.py"

,tipo SQLite,dtype,nulos,% nulos,qué es / para qué sirve
columna,,,,,
recommendationid,TEXT,object,0,0.0,Id único de la reseña en Steam
appid,INT,int64,0,0.0,Id del juego; une la reseña con su juego
num_games_owned,INT,int64,0,0.0,Juegos en la biblioteca del autor; 0 = perfil privado
num_reviews,INT,int64,0,0.0,Reseñas escritas por el autor; posterior a la compra
playtime_forever,INT,int64,0,0.0,Minutos jugados en total hasta la descarga
playtime_last_two_weeks,INT,int64,0,0.0,Minutos jugados en las dos semanas antes de descargar
playtime_at_review,INT,int64,0,0.0,Minutos jugados al reseñar; define la señal (< 120)
last_played,INT,int64,0,0.0,"Última vez que el autor jugó, como marca de tiempo"
idioma,TEXT,object,0,0.0,Idioma que reporta Steam; la ingesta pidió inglés


In [11]:
esquema_juegos = ex.esquema(rutas["data-v1"], "juegos", juegos_v1)
display(esquema_juegos)
assert (esquema_juegos[ex.QUE_ES] != "").all(), "una columna de juegos no está en backend/analisis/diccionario.py"

,tipo SQLite,dtype,nulos,% nulos,qué es / para qué sirve
columna,,,,,
appid,INT,int64,0,0.00,Id del juego en Steam; llave de la tabla
nombre,TEXT,object,0,0.00,Nombre del juego como aparece en la tienda
tipo,TEXT,object,0,0.00,Tipo de producto en Steam; aquí siempre «game»
fecha_lanzamiento,TEXT,object,0,0.00,Fecha de lanzamiento que muestra la tienda
proximamente,INT,int64,0,0.00,Si la tienda lo marca como «próximamente»
es_gratis,INT,int64,0,0.00,Si el juego es gratuito; entra al modelo
precio_inicial,INT,float64,4,4.82,"Precio sin descuento, en centavos de peso"
precio_final,INT,float64,4,4.82,Precio con descuento al corte; entra al modelo
descuento,INT,float64,4,4.82,"Descuento al corte de datos, en %; entra al modelo"


**Resultado.** En `resenas`, las columnas que usa el modelo no tienen nulos. Los nulos de `juegos` están en
el precio y en la nota de la crítica: los juegos sin nota son un dato del modelo (`metacritic_disponible`),
y los de pago sin precio se revisan en 1.4. `num_games_owned == 0` no es un nulo: es la bandera de un perfil
privado, y no se imputa como biblioteca vacía.

## 1.3 Llaves

In [12]:
tabla_llaves = ex.llaves(juegos_v1, resenas_v1)
display(tabla_llaves)
assert (tabla_llaves["casos"] == 0).all(), "hay una llave rota"

,casos
chequeo,
recommendationid repetido,0
appid de reseñas que no está en juegos,0
juegos sin reseñas,0
appid repetido en juegos,0


**Resultado.** Cada reseña tiene un `recommendationid` único, todas apuntan a un juego que existe y ningún
juego se quedó sin reseñas.

## 1.4 Rangos y consistencia

In [13]:
casos = ex.casos_de_rango(juegos_v1, resenas_v1)
display(ex.resumen_de_rango(casos))
for regla in ex.REGLAS_DURAS:
    assert casos[regla].empty, f"falla la regla «{regla}»"

,casos,tipo
regla,,
playtime_at_review ≥ 0,0,assert
"voted_up ∈ {0, 1}",0,assert
creada ≤ editada ≤ descargada,0,assert
precio ≥ 0,0,assert
descuento entre 0 y 100,0,assert
Metacritic entre 0 y 100 o nulo,0,assert
playtime_at_review ≤ playtime_forever,1,tabla de casos
es_gratis coherente con el precio,2,tabla de casos


In [14]:
# Minutos al reseñar mayores que el total: pasa cuando Steam actualiza un contador antes que el otro.
fuera_de_orden = casos["playtime_at_review ≤ playtime_forever"]
display(fuera_de_orden[["recommendationid", "appid", "playtime_at_review", "playtime_forever", "voted_up"]])
# La señal sale igual con cualquiera de los dos contadores: no se corrige.
assert (ex.senal(fuera_de_orden) == ex.senal(fuera_de_orden.assign(playtime_at_review=fuera_de_orden["playtime_forever"]))).all()

,recommendationid,appid,playtime_at_review,playtime_forever,voted_up
113459,193426847,1294810,229,213,1


In [15]:
display(casos["es_gratis coherente con el precio"][["appid", "nombre", "es_gratis", "precio_final"]])

,appid,nombre,es_gratis,precio_final
19,271590,Grand Theft Auto V Legacy,0,NaN
51,1063730,New World: Aeternum,0,NaN


**Resultado.** Las reglas duras se cumplen: minutos no negativos, pulgar en {0, 1}, fechas dentro de la
ventana de descarga, precio, descuento y nota en rango. Quedan dos tablas de casos:

- Una reseña con más minutos al reseñar que en total. La señal no cambia con ninguno de los dos contadores,
  así que se deja.
- **GTA V Legacy y New World: Aeternum** son de pago pero llegaron sin precio, y el modelo lo lee como 0.
  Es una **decisión ya tomada**, no se corrige aquí: su efecto está medido en `01_modelo_riesgo.ipynb`, sección 4
  («Limitación conocida: juegos de pago con el precio imputado como 0»). Lo que sí cambia es cómo se
  explica: la ficha mostrará ese factor, con el aviso «Estimación menos confiable» junto al veredicto, en
  lugar de ocultarlo (se implementa en la ronda «explicar el riesgo»).

## 1.5 Fuga y duplicados

Un texto idéntico en dos reseñas puede ser una copia (la misma persona, una plantilla, un meme) o pura
coincidencia («good game»). Para los modelos de texto importa sobre todo la copia **entre juegos
distintos**: con GroupKFold, el mismo texto podría quedar en entrenamiento y en validación.

In [16]:
todas = ex.copias_de_texto(resenas_v1)
largas = ex.copias_de_texto(resenas_v1, li.PALABRAS_DE_UNA_COPIA)
conteos = ["sobrantes en el mismo juego", "textos en 2+ juegos", "filas con texto de 2+ juegos"]
display(pd.DataFrame({"cualquier largo": [todas[c] for c in conteos],
                      f"{li.PALABRAS_DE_UNA_COPIA}+ palabras": [largas[c] for c in conteos]}, index=conteos))

,cualquier largo,8+ palabras
sobrantes en el mismo juego,12146,359
textos en 2+ juegos,2282,57
filas con texto de 2+ juegos,24049,188


In [17]:
print(f"Copias de {li.PALABRAS_DE_UNA_COPIA}+ palabras dentro del mismo juego:")
display(ex.ejemplos(largas["ejemplos en el mismo juego"], juegos_v1))

Copias de 8+ palabras dentro del mismo juego:


,nombre,voted_up,playtime_at_review,texto
0,New World: Aeternum,0,1007,"If companies can terminate a game at anytime, then I should be allowed a refund at anytime."
1,New World: Aeternum,0,3290,"If companies can terminate a game at anytime, then I should be allowed a refund at anytime"
2,New World: Aeternum,0,3519,"If companies can terminate a game at anytime, then I should be allowed a refund at anytime."


In [18]:
print(f"Copias de {li.PALABRAS_DE_UNA_COPIA}+ palabras entre juegos distintos:")
display(ex.ejemplos(largas["ejemplos entre juegos"], juegos_v1))

Copias de 8+ palabras entre juegos distintos:


,nombre,voted_up,playtime_at_review,texto
0,Half-Life 2,1,653,one of the best games ive ever played
1,Portal,1,224,"It's a Portal game. Challenging yet fun puzzles, funny robot voices. What hasn't already been said about it."
2,The Binding of Isaac: Rebirth,1,8386,this is the best game i have ever played


In [19]:
con_plantilla = ex.plantillas(resenas_v1)
print(f"reseñas plantilla: {len(con_plantilla):,} ({len(con_plantilla) / len(resenas_v1):.2%}); "
      f"con señal: {ex.senal(con_plantilla).sum()}")
display(ex.ejemplos(con_plantilla, juegos_v1))

reseñas plantilla: 192 (0.15%); con señal: 3


,nombre,voted_up,playtime_at_review,texto
0,No Man's Sky,1,16667,---{ Graphics }---\r\n☐ You forget what reality is\r\n☑ Beautiful\r\n☐ Good\r\n☐ Decent\r\n☐ Bad\r\n☐ Don‘t look too long at it\r\n☐ MS-DOS\r\n\r\n---{ Gameplay }---\r\n☑ Very good\r\n☐ Good\r\n☐ It's just gameplay\r\n☐ Mehh\r...
1,Battlefield™ 2042,0,2962,---{ Graphics }---\n☐ You forget what reality is\n☐ Beautiful\n▣ Good\n☐ Decent\n☐ Bad\n☐ Don‘t look too long at it\n☐ MS-DOS\n\n---{ Gameplay }---\n☐ Very good\n▣ Good\n☐ It's just gameplay\n☐ Mehh\n☐ Watch paint dry instead\...
2,Factorio,1,29575,---{ Graphics }---\r\n☐ You forget what reality is\r\n☐ Beautiful\r\n☑ Good\r\n☐ Decent\r\n☐ Bad\r\n☐ Don‘t look too long at it\r\n☐ MS-DOS\r\n\r\n---{ Gameplay }---\r\n☑ Very good\r\n☐ Good\r\n☐ It's just gameplay\r\n☐ Mehh\r...


**Resultado.** Los textos cortos se repiten por miles porque mucha gente escribe lo mismo; no son copias. Con
8 palabras o más, las repeticiones son pocas: frases hechas («one of the best games I've ever played»),
memes, plantillas, campañas de protesta copiadas y la misma reseña publicada en varios juegos. La limpieza (2.1) quita esas copias para los modelos de texto. Las reseñas plantilla («☐ Bad ☑ Good»)
pesan poco y casi nunca llevan señal; se marcan en 2.5.

## 1.6 Partición congelada

Los folds de GroupKFold se asignan por juego. La partición está congelada en el repo
(`backend/referencias/particion_gkf_data-v1.csv`) y toda evaluación la lee de ahí, en este notebook y en
`01_modelo_riesgo.ipynb`. No se recalcula: casi todos los juegos empatan en el tope de la ingesta (1,500
reseñas) y cada versión de scikit-learn desempata distinto. Aquí se comprueba que el CSV es una partición
válida de los mismos juegos que leen 00 y 01, con su firma, y se compara con lo que arma el GroupKFold de
la versión instalada, solo para avisar si difiere.

In [20]:
folds = ex.leer_particion()
chequeos = ex.chequeo_de_particion(folds, {"data-v1 (base con texto)": df_v1["appid"], "extracto de 01": parquet_v1["appid"]})
display(chequeos)
assert chequeos["ok"].all(), "el CSV no es una partición válida de los juegos de data-v1"

groupkfold = ex.groupkfold_de_esta_version(df_v1, folds)
if groupkfold["misma partición"]:
    print(f"El GroupKFold de scikit-learn {groupkfold['scikit-learn']} arma la misma partición que el CSV.")
else:
    print(f"Aviso: el GroupKFold de scikit-learn {groupkfold['scikit-learn']} reparte los juegos de otra forma "
          f"(coincide en {groupkfold['juegos con el mismo fold']} de {groupkfold['juegos']} juegos): "
          f"{groupkfold['juegos empatados']} de los {groupkfold['juegos']} tienen exactamente "
          f"{groupkfold['reseñas del empate']:,} reseñas y el desempate cambió entre versiones. "
          "Este notebook sigue usando la partición congelada del CSV.")

,valor,esperado,ok
chequeo,,,
appids en el CSV,83,83,True
folds,5,5,True
juegos en más de un fold,0,0,True
appids de data-v1 (base con texto) fuera del CSV,0,0,True
appids de extracto de 01 fuera del CSV,0,0,True
appids del CSV fuera de los datos,0,0,True
firma,416aa8243b570f05f40aa238cb9730faa67d41ecd3f79da7db56558028bb7493,416aa8243b570f05f40aa238cb9730faa67d41ecd3f79da7db56558028bb7493,True


Aviso: el GroupKFold de scikit-learn 1.6.1 reparte los juegos de otra forma (coincide en 12 de 83 juegos): 73 de los 83 tienen exactamente 1,500 reseñas y el desempate cambió entre versiones. Este notebook sigue usando la partición congelada del CSV.


In [21]:
display(ex.resumen_por_fold(resenas_v1, folds))

,juegos,reseñas,Y=1,prevalencia
fold,,,,
0,16,24198,612,0.0253
1,16,24198,453,0.0187
2,17,24878,635,0.0255
3,17,25599,604,0.0236
4,17,25099,410,0.0163


In [22]:
copias_cruzadas = ex.copias_en_folds_distintos(resenas_v1, largas["claves entre juegos"], folds)
print(f"textos de {li.PALABRAS_DE_UNA_COPIA}+ palabras copiados entre juegos: {largas['textos en 2+ juegos']}; "
      f"en folds distintos: {copias_cruzadas}")

textos de 8+ palabras copiados entre juegos: 57; en folds distintos: 52


**Resultado.** 00 y 01 validan con la misma partición: los dos la leen del CSV del repo, que cubre
exactamente los juegos de la base con texto y del extracto de 01, y su firma es la registrada. Si el
GroupKFold de otra versión de scikit-learn la arma distinto, la celda lo avisa y el notebook sigue con la
congelada; cuánto cambia el PR-AUC con otra partición está en la sección 3.2 y en
`docs/evidencia/particion-alternativa.json`. La señal no se reparte parejo entre folds, y esa es una de las razones
de que el PR-AUC varíe de un fold a otro. Casi todas las copias entre juegos caen en folds distintos: por eso
la limpieza las quita antes de entrenar modelos de texto.

# Bloque 2 · Limpieza

La limpieza produce el conjunto para los **modelos de texto** (Parte A: predecir el pulgar desde el texto;
Parte B: los motivos). **No se aplica al modelo de riesgo**: ese se entrena con data-v1 tal cual, para que
`verificar_bandas.py` siga comparando contra las mismas bandas.

Cada regla es una función de `backend/analisis/limpieza.py`. Después de cada una se ven los conteos de antes y
después y tres textos reales. Solo dos reglas quitan filas; las demás marcan una columna para no cambiar la
población que se estudia.

### 2.1 Duplicados exactos (8 palabras o más)

In [23]:
limpio, paso = li.quitar_duplicados_exactos(resenas_v1)
pasos = [paso]
print(f"{paso.antes:,} → {paso.despues:,} filas ({len(paso.afectadas):,} quitadas)")
display(ex.ejemplos(paso.afectadas, juegos_v1))

123,972 → 123,489 filas (483 quitadas)


,nombre,voted_up,playtime_at_review,texto
0,New World: Aeternum,0,2252,"If companies can terminate a game at anytime, then I should be allowed a refund at anytime.\n"
1,Cyberpunk 2077,1,6005,I really want to stay at your house
2,New World: Aeternum,0,21207,"If companies can terminate a game at anytime, then I should be allowed a refund at anytime."


### 2.2 Reseñas vacías

In [24]:
limpio, paso = li.quitar_vacias(limpio)
pasos.append(paso)
print(f"{paso.antes:,} → {paso.despues:,} filas ({len(paso.afectadas):,} quitadas); con señal: {ex.senal(paso.afectadas).sum()}")
display(ex.ejemplos(paso.afectadas, juegos_v1))

123,489 → 122,944 filas (545 quitadas); con señal: 1


,nombre,voted_up,playtime_at_review,texto
0,Valheim,1,3014,
1,Europa Universalis IV,1,221528,
2,Hades,1,971,


### 2.3 Reseñas cortas (menos de 3 palabras): se marcan, no se quitan

Quitarlas cambiaría la población: son casi un cuarto de las reseñas. En la Parte B quedan fuera del conjunto
de referencia (con dos palabras no hay motivo que leer); en la Parte A se entrena con todas y el resultado se
reporta por longitud.

In [25]:
limpio, paso = li.marcar_cortas(limpio)
pasos.append(paso)
cortas = limpio["es_corta"]
print(f"cortas: {cortas.sum():,} de {len(limpio):,} ({cortas.mean():.1%}); "
      f"con señal: {(cortas & ex.senal(limpio)).sum()} de {ex.senal(limpio).sum():,} ({(cortas & ex.senal(limpio)).sum() / ex.senal(limpio).sum():.1%})")
display(ex.ejemplos(paso.afectadas, juegos_v1))

cortas: 27,944 de 122,944 (22.7%); con señal: 235 de 2,706 (8.7%)


,nombre,voted_up,playtime_at_review,texto
0,Dead Cells,1,2311,!
1,Persona 5 Royal,1,2347,Joker
2,Hearts of Iron IV,1,10962,fun


### 2.4 Reseñas en otro idioma: se marcan, no se quitan

Steam las etiqueta como inglés porque la ingesta pide `language=english`, pero una parte está escrita en otro
idioma. `lingua` decide solo cuando está segura; lo dudoso (textos muy cortos, gibberish) queda como
«indeterminado» y no se marca.

In [26]:
limpio, paso = li.marcar_no_ingles(limpio)
pasos.append(paso)
otro_idioma = limpio["no_ingles"]
negativas_otro_idioma = int((otro_idioma & (limpio["voted_up"] == 0)).sum())
print(f"en otro idioma: {otro_idioma.sum():,} ({otro_idioma.mean():.2%}); negativas: {negativas_otro_idioma}; "
      f"con señal: {(otro_idioma & ex.senal(limpio)).sum()}")
display(limpio.loc[otro_idioma, "idioma_detectado"].value_counts().head(8).rename("reseñas").to_frame())
display(ex.ejemplos(paso.afectadas, juegos_v1))

en otro idioma: 1,637 (1.33%); negativas: 223; con señal: 36


,reseñas
idioma_detectado,
russian,400
spanish,327
portuguese,232
turkish,120
polish,66
arabic,59
german,59
chinese,58


,nombre,voted_up,playtime_at_review,texto
0,Dead by Daylight,0,11331,"[h1]Empresa completamente incapaz de não foder com o próprio jogo.[/h1] Essa última atualização da DENUVO deixou o jogo rodando lixosamente mesmo em máquinas boas, mesmo assim não temos NENHUMA resposta da Behavior sobre…"
1,Diablo® IV,1,974,Mega fajna wciągająca gra
2,Slay the Spire,1,5379,АААААААААААААААААА БЛЯЯЯЯЯЯЯЯЯЯЯЯЯЯЯЯЯЯЯЯ


In [27]:
en_ingles = 1 - otro_idioma.mean()
print(f"Decisión: all-MiniLM-L6-v2 (inglés). El {en_ingles:.1%} de las reseñas está en inglés o no se pudo determinar; "
      f"el modelo multilingüe es ~3–5 veces más lento (estimado: 12 capas y 118M parámetros contra 6 y 22M) "
      f"para cubrir el {otro_idioma.mean():.1%} restante.")
print(f"Parte A: con {negativas_otro_idioma} negativas en otro idioma "
      f"{'alcanza' if negativas_otro_idioma >= 50 else 'no alcanza'} para reportar su PR-AUC aparte (mínimo 50).")

Decisión: all-MiniLM-L6-v2 (inglés). El 98.7% de las reseñas está en inglés o no se pudo determinar; el modelo multilingüe es ~3–5 veces más lento (estimado: 12 capas y 118M parámetros contra 6 y 22M) para cubrir el 1.3% restante.
Parte A: con 223 negativas en otro idioma alcanza para reportar su PR-AUC aparte (mínimo 50).


**Qué se hace con ellas.** En la Parte B quedan **fuera** del conjunto de referencia de 300 y se reportan como
limitación: el modelo de embeddings está entrenado para inglés. En la Parte A se quedan en el entrenamiento, y
su PR-AUC se reporta aparte.

### 2.5 Reseñas plantilla: se marcan, no se quitan

In [28]:
limpio, paso = li.marcar_plantillas(limpio)
pasos.append(paso)
print(f"plantillas: {limpio['es_plantilla'].sum():,} ({limpio['es_plantilla'].mean():.2%})")
display(ex.ejemplos(paso.afectadas, juegos_v1))

plantillas: 191 (0.16%)


,nombre,voted_up,playtime_at_review,texto
0,Persona 5 Royal,1,4033,---{ Graphics }---\n☐ You forget what reality is\n☑ Beautiful\n☐ Good\n☐ Decent\n☐ Bad\n☐ Don't look too long at it\n☐ MS-DOS\n\n---{ Gameplay }---\n☑ Very good\n☐ Good\n☐ It's just gameplay\n☐ Mehh\n☐ Watch paint dry instead\...
1,Factorio,1,29575,---{ Graphics }---\r\n☐ You forget what reality is\r\n☐ Beautiful\r\n☑ Good\r\n☐ Decent\r\n☐ Bad\r\n☐ Don‘t look too long at it\r\n☐ MS-DOS\r\n\r\n---{ Gameplay }---\r\n☑ Very good\r\n☐ Good\r\n☐ It's just gameplay\r\n☐ Mehh\r...
2,The Last of Us™ Parte I,1,613,---{ Graphics }---\n☐ You forget what reality is\n☑ Beautiful\n☐ Good\n☐ Decent\n☐ Bad\n☐ MS Paint\n\n---{ Gameplay }---\n☐ Masterpiece\n☐ Very good\n☐ Good\n☑ Good but repetitive\n☐ Mehh\n☐ Watch paint dry instead\n\n---{ Sto...


### 2.6 Normalización del texto

Minúsculas, sin BBCode (`[h1]`, `[spoiler]`), URLs ni casillas. Las contracciones se abren para que la
negación quede como palabra suelta («don't» → «do not»): sin eso, «not fun» y «fun» se leerían igual. Las
negaciones también se conservan en la lista de palabras vacías que se usa en 3.7.

In [29]:
limpio, paso = li.normalizar(limpio)
pasos.append(paso)
con_negacion = limpio[limpio["texto"].fillna("").str.contains("n't|dont|cant|wont|isnt", case=False)]
display(ex.ejemplos(con_negacion, juegos_v1, columnas=("texto", "texto_norm")))
assert {"not", "no", "never", "nor"}.isdisjoint(li.STOPWORDS_SIN_NEGACIONES)

,nombre,voted_up,playtime_at_review,texto,texto_norm
0,Project Zomboid,1,6206,"First, don't let my mere hundreds of hours of PZ hours here fool ya lads. I have a 1 in-game year character in a 2 IRL hour per 1 IG day server before (730+ IRL hours) plus other characters reaching hundreds of IG days. …",first do not let my mere hundreds of hours of pz hours here fool ya lads i have a 1 in game year character in a 2 irl hour per 1 ig day server before 730 irl hours plus other characters reaching hundreds of ig days secon…
1,Battlefield™ 2042,0,22,"The game looks like mobile scrappy game from 2018 XD, yet i keep experiencing freezes every few seconds, didnt even bother to figure it out, thank god i bought this for 4eur or smth, cant belive they even put out product…",the game looks like mobile scrappy game from 2018 xd yet i keep experiencing freezes every few seconds did not even bother to figure it out thank god i bought this for 4eur or smth can not belive they even put out produc…
2,Risk of Rain 2,0,1187,poop game all them builds is boring i hate the difficulty progression being tied to time spent. looting is annoying most characters are unsatisfying to move around with and waiting for abilities is annoying i cant play t…,poop game all them builds is boring i hate the difficulty progression being tied to time spent looting is annoying most characters are unsatisfying to move around with and waiting for abilities is annoying i can not play…


### Bitácora

In [30]:
display(li.bitacora(pasos))

,regla,filas antes,filas después,afectadas,motivo
0,quitar_duplicados_exactos,123972,123489,483,texto idéntico de 8+ palabras; queda la más antigua
1,quitar_vacias,123489,122944,545,sin texto no hay nada que leer
2,marcar_cortas,122944,122944,27944,"menos de 3 palabras; se marcan, no se quitan"
3,marcar_no_ingles,122944,122944,1637,"otro idioma con confianza (lingua); se marcan, no se quitan"
4,marcar_plantillas,122944,122944,191,"reseña plantilla («☐ Bad ☑ Good»); se marcan, no se quitan"
5,normalizar_texto,122944,122944,94303,"minúsculas, sin BBCode ni URLs, negaciones como palabra; agrega texto_norm"


### No aplicada al modelo congelado

¿Qué le pasaría al modelo de riesgo si se entrenara con estas reglas? Se mide sin aplicarlo, con la
partición congelada antes y después: así la diferencia es de las filas quitadas y no de cómo se reparten
los juegos entre folds.

In [31]:
sin_cortas = resenas_v1[~li.marcar_cortas(resenas_v1)[0]["es_corta"]]
impacto = pd.DataFrame.from_dict(orient="index", data={
    "quitar duplicados": ex.impacto_en_modelo_congelado(df_v1, ex.con_juego(li.quitar_duplicados_exactos(resenas_v1)[0], juegos_v1), folds),
    "quitar vacías": ex.impacto_en_modelo_congelado(df_v1, ex.con_juego(li.quitar_vacias(resenas_v1)[0], juegos_v1), folds),
    "quitar cortas (hipótesis)": ex.impacto_en_modelo_congelado(df_v1, ex.con_juego(sin_cortas, juegos_v1), folds),
})
display(impacto)

,filas quitadas,Y=1 quitadas,PR-AUC antes,PR-AUC después (partición congelada),juegos que cambian de nivel al reentrenar
quitar duplicados,483,7,0.0694,0.0695,1
quitar vacías,545,1,0.0694,0.0696,0
quitar cortas (hipótesis),28489,236,0.0694,0.0800,0


In [32]:
duplicados = impacto.loc["quitar duplicados"]
print(f"Quitar duplicados, con la partición congelada: PR-AUC {duplicados['PR-AUC antes']:.4f} → "
      f"{duplicados['PR-AUC después (partición congelada)']:.4f}.")
assert abs(duplicados["PR-AUC después (partición congelada)"] - duplicados["PR-AUC antes"]) < 0.005

Quitar duplicados, con la partición congelada: PR-AUC 0.0694 → 0.0695.


**Qué vemos.** Con la partición congelada, quitar duplicados o vacías deja el PR-AUC prácticamente igual.
Quitar las cortas sí lo mueve: cambia la población, no solo el ruido. Si la partición se recalculara, el
PR-AUC se movería además solo por cómo quedan repartidos los juegos (la sección 3.2 mide cuánto); por eso
toda comparación usa la congelada.

**Qué decidimos.** El modelo de riesgo se queda como está (**no aplicada al modelo congelado**), y la
partición se fija por firma para que cualquier comparación futura use los mismos folds.

### El conjunto limpio

In [33]:
ARTEFACTOS = Path("extracto_eda")
ARTEFACTOS.mkdir(exist_ok=True)
limpio.to_parquet(ARTEFACTOS / "limpio_data-v1.parquet", index=False)
firma_limpio = li.firma_del_conjunto(limpio)
print(f"{len(limpio):,} reseñas → {ARTEFACTOS / 'limpio_data-v1.parquet'}")
print(f"firma del contenido: {firma_limpio}")
assert firma_limpio == li.FIRMA_LIMPIO_DATA_V1, "el conjunto limpio cambió: revisar las reglas o la versión de lingua"

122,944 reseñas → extracto_eda/limpio_data-v1.parquet
firma del contenido: cb1e4b06fdf5a0264471f722446ac73bd06e28db30653819b8f1773c58e3f41e


La firma es del contenido (ids, texto normalizado y banderas), no del archivo: los bytes de un parquet cambian
con la versión de pyarrow aunque las filas sean las mismas. Los modelos de texto (02) rehacen este conjunto
con las mismas funciones y comprueban la misma firma.

# Bloque 3 · Exploración

Cada sección empieza con una pregunta y termina con **Qué vemos** y **Qué decidimos**.

## 3.1 Cobertura

**Pregunta:** ¿cuántas reseñas hay de cada juego y qué periodo cubren?

In [34]:
cobertura = ex.cobertura_por_juego(resenas_v1, juegos_v1)
display(cobertura["reseñas"].describe().round(0).to_frame().T)
en_el_tope = (cobertura["reseñas"] >= ex.TOPE_DE_LA_INGESTA).sum()
print(f"juegos que llegaron al tope de la ingesta ({ex.TOPE_DE_LA_INGESTA:,}): {en_el_tope} de {len(cobertura)}")
dias = cobertura["días cubiertos"]
print(f"días que cubre la muestra de cada juego: de {dias.min()} a {dias.max():,} (mediana {dias.median():.0f})")
fechas = pd.to_datetime(resenas_v1["timestamp_created"], unit="s")
recientes = (fechas >= fechas.max() - pd.Timedelta(days=365)).mean()
print(f"reseñas publicadas en el último año antes de la descarga: {recientes:.1%}")
assert recientes > 0.8

,count,mean,std,min,25%,50%,75%,max
reseñas,83.0,1494.0,110.0,682.0,1500.0,1500.0,1500.0,1599.0


juegos que llegaron al tope de la ingesta (1,500): 81 de 83
días que cubre la muestra de cada juego: de 4 a 1,252 (mediana 64)
reseñas publicadas en el último año antes de la descarga: 94.9%


In [35]:
fig = px.scatter(cobertura.reset_index(), x="días cubiertos", y="reseñas", hover_name="nombre", log_x=True,
                 title="Reseñas en la muestra contra días que cubren, por juego")
fig.update_traces(marker={"color": ex.PALETA["AZUL"], "size": 8})
fig.add_hline(y=ex.TOPE_DE_LA_INGESTA, line_dash="dot", line_color=ex.PALETA["GRIS"],
              annotation_text="tope de la ingesta", annotation_position="bottom right")
fig.show()

In [36]:
por_mes = pd.to_datetime(resenas_v1["timestamp_created"], unit="s").dt.to_period("M").value_counts().sort_index()
fig = px.bar(x=por_mes.index.to_timestamp(), y=por_mes.values, labels={"x": "mes", "y": "reseñas"},
             title="Reseñas por mes de publicación")
fig.update_traces(marker_color=ex.PALETA["AZUL"])
fig.update_xaxes(tickformat="%Y-%m")
fig.show()

In [37]:
print(f"Limitación de la fuente: el {recientes:.1%} de las reseñas es del último año antes de la descarga, y "
      f"{en_el_tope} de {len(cobertura)} juegos llegan al tope de {ex.TOPE_DE_LA_INGESTA:,} reseñas recientes. "
      "En los juegos viejos, la señal temprana es la de quienes llegan tarde.")

Limitación de la fuente: el 94.9% de las reseñas es del último año antes de la descarga, y 81 de 83 juegos llegan al tope de 1,500 reseñas recientes. En los juegos viejos, la señal temprana es la de quienes llegan tarde.


**Qué vemos.** La ingesta pide las reseñas más recientes (`filter=recent`) hasta un tope por juego, y casi
todos los juegos lo alcanzan. Un juego popular llena el tope en pocos días; uno de nicho o ya abandonado
necesita años. Por eso casi todas las reseñas son del último año, y la muestra de cada juego describe su
momento actual, no su lanzamiento.

**Qué decidimos.** Se reporta como **limitación de la fuente**, con sus cifras en la celda de arriba: la tasa
de señal de un juego es la de sus reseñas recientes. En los juegos viejos, la señal temprana viene de
**jugadores que llegan tarde** (compran años después del lanzamiento, en una oferta o tras una
actualización), no de quienes lo compraron al salir. No se corrige con pesos, porque no hay de dónde saber
cómo era la mezcla en otra época.

## 3.2 Por qué GroupKFold

**Pregunta:** ¿qué pasaría si se validara mezclando reseñas del mismo juego en entrenamiento y validación?

In [38]:
por_juego = ex.tasa_por_juego(resenas_v1, juegos_v1)
dispersion = ex.sobredispersion(por_juego)
print(f"tasa de señal por juego: de {por_juego['tasa'].min():.2%} a {por_juego['tasa'].max():.2%}")
print(f"la tasa varía {dispersion:.0f} veces más entre juegos de lo que variaría si todos tuvieran la misma")
assert dispersion > 10

tasa de señal por juego: de 0.13% a 27.13%
la tasa varía 107 veces más entre juegos de lo que variaría si todos tuvieran la misma


In [39]:
orden = por_juego.sort_values("tasa").reset_index()
fig = go.Figure(go.Scatter(
    x=orden.index, y=orden["tasa"], mode="markers", text=orden["nombre"], marker={"color": ex.PALETA["AZUL"]},
    error_y={"type": "data", "symmetric": False, "array": orden["ic_alto"] - orden["tasa"],
             "arrayminus": orden["tasa"] - orden["ic_bajo"], "color": ex.PALETA["GRIS"]},
))
fig.add_hline(y=ex.senal(resenas_v1).mean(), line_dash="dot", line_color=ex.PALETA["ROJO"],
              annotation_text="prevalencia global")
fig.update_layout(title="Tasa de señal por juego, con intervalo de Wilson al 95 %",
                  xaxis_title="juegos, de menor a mayor tasa", yaxis_title="tasa de señal", yaxis_tickformat=".0%")
fig.show()

In [40]:
constantes = ex.variables_constantes_por_juego(df_v1)
display(constantes.to_frame())
assert (constantes == 0).all(), "una variable del modelo cambia dentro de un juego"

,juegos con más de un valor
es_gratis,0
log_precio_final,0
descuento,0
metacritic_disponible,0
metacritic,0


In [41]:
validaciones = ex.kfold_contra_groupkfold(df_v1)
display(validaciones.round(4))
assert validaciones.loc["KFold aleatorio (fuga)", "PR-AUC media"] > validaciones.loc["GroupKFold por appid", "PR-AUC media"]

,PR-AUC media,std entre folds
validación,,
KFold aleatorio (fuga),0.0799,0.0032
GroupKFold por appid,0.0694,0.0415


In [42]:
particiones = ex.sensibilidad_a_la_particion(df_v1)
congelada = ex.pr_auc_con_particion(df_v1, folds).mean()
print(f"PR-AUC con 20 particiones GroupKFold distintas: media {particiones.mean():.4f}, "
      f"mínimo {particiones.min():.4f}, máximo {particiones.max():.4f}. Con la partición congelada: {congelada:.4f}.")
assert particiones.min() <= congelada <= particiones.max()

PR-AUC con 20 particiones GroupKFold distintas: media 0.0694, mínimo 0.0477, máximo 0.0873. Con la partición congelada: 0.0694.


**Qué vemos.** La tasa de señal cambia mucho de un juego a otro, bastante más de lo que explicaría el azar, y
las cinco variables del modelo son idénticas dentro de cada juego. Con folds aleatorios, el modelo ve el
juego en entrenamiento y en validación: su PR-AUC sale más alto y con una desviación mínima, porque en
realidad está reconociendo juegos, no generalizando.

Además, solo cambiando qué juegos caen en cada fold, el PR-AUC se mueve en un rango amplio (la última celda):
una diferencia menor que ese rango no dice nada de los datos.

**Qué decidimos.** Se valida con **GroupKFold por appid**: el modelo se mide con juegos que no vio. El
KFold aleatorio de arriba es solo para mostrar la fuga; no se usa para decidir nada. Y la partición se
**congela**: toda comparación de este notebook usa los mismos folds.

## 3.3 La señal

**Pregunta:** ¿qué tan rara es la señal, dónde cae el corte de 120 minutos y por qué se mide con PR-AUC?

In [43]:
prevalencia = ex.senal(resenas_v1).mean()
print(f"prevalencia de la señal: {prevalencia:.2%} ({ex.senal(resenas_v1).sum():,} de {len(resenas_v1):,})")
exactitud = ex.exactitud_contra_pr_auc(df_v1)
display(exactitud.round(4))
# El PR-AUC de un modelo que no sabe nada es la prevalencia.
assert abs(exactitud.loc["siempre 0 (trivial)", "PR-AUC (GroupKFold)"] - prevalencia) < 0.005

prevalencia de la señal: 2.19% (2,714 de 123,972)


,exactitud,PR-AUC (GroupKFold)
modelo,,
siempre 0 (trivial),0.9781,0.0219
modelo del juego,NaN,0.0694


In [44]:
pulgar = resenas_v1["voted_up"].map({1: "positiva", 0: "negativa"})
antes_de_120 = (resenas_v1["playtime_at_review"] < ex.UMBRAL_REEMBOLSO).groupby(pulgar).mean()
print(f"escritas antes de 120 min: negativas {antes_de_120['negativa']:.1%}, positivas {antes_de_120['positiva']:.1%}")
assert antes_de_120["negativa"] > 3 * antes_de_120["positiva"]

escritas antes de 120 min: negativas 15.6%, positivas 2.0%


In [45]:
# ¿Hay un escalón en 120? Reseñas por minuto justo antes y justo después, y en 180, donde la figura muestra uno.
minutos = resenas_v1["playtime_at_review"]
ventanas = {"110 a 119": (110, 120), "120 a 129": (120, 130), "170 a 179": (170, 180), "180 a 189": (180, 190)}
por_minuto = pd.DataFrame({nombre: minutos.between(desde, hasta - 1).groupby(pulgar).sum() / (hasta - desde)
                           for nombre, (desde, hasta) in ventanas.items()}).T.rename_axis("minutos al reseñar")
display(por_minuto.round(1))
en_120 = por_minuto.loc["120 a 129"] / por_minuto.loc["110 a 119"]
en_180 = por_minuto.loc["180 a 189"] / por_minuto.loc["170 a 179"]
juegos_180 = resenas_v1.loc[minutos.between(180, 189) & (resenas_v1["voted_up"] == 1), "appid"].nunique()
print(f"Reseñas por minuto justo antes y justo después de 120: positivas {por_minuto.loc['110 a 119', 'positiva']:.1f} → "
      f"{por_minuto.loc['120 a 129', 'positiva']:.1f}, negativas {por_minuto.loc['110 a 119', 'negativa']:.1f} → "
      f"{por_minuto.loc['120 a 129', 'negativa']:.1f}. En 180: positivas {por_minuto.loc['170 a 179', 'positiva']:.1f} → "
      f"{por_minuto.loc['180 a 189', 'positiva']:.1f} ({en_180['positiva']:.1f} veces, en {juegos_180} juegos), "
      f"negativas {por_minuto.loc['170 a 179', 'negativa']:.1f} → {por_minuto.loc['180 a 189', 'negativa']:.1f}.")
assert en_120.between(0.8, 1.25).all(), "apareció un escalón en 120"
assert en_180["positiva"] > 3 * en_180["negativa"], "el salto de las positivas en 180 ya no está"

voted_up,negativa,positiva
minutos al reseñar,,
110 a 119,11.9,17.4
120 a 129,12.2,18.6
170 a 179,9.9,17.6
180 a 189,12.4,81.7


Reseñas por minuto justo antes y justo después de 120: positivas 17.4 → 18.6, negativas 11.9 → 12.2. En 180: positivas 17.6 → 81.7 (4.6 veces, en 81 juegos), negativas 9.9 → 12.4.


In [46]:
# Intervalos del mismo ancho en escala log, con un borde justo en 180 para que el salto se vea donde está.
# Ninguno cae en 120. Se dibuja en escalones desde los bordes reales de cada intervalo.
log_minutos = np.log10(minutos + 1)
ancho = np.log10(181) / 22
bordes = np.arange(0, log_minutos.max() + ancho, ancho)
assert not np.isclose(10 ** bordes - 1, ex.UMBRAL_REEMBOLSO, atol=1).any(), "un borde cae en 120"
barras = ex.histograma_por_grupo(log_minutos, pulgar, bordes)
fig = go.Figure()
for grupo, color in (("negativa", ex.PALETA["ROJO"]), ("positiva", ex.PALETA["VERDE_OSC"])):
    porcentaje = barras.loc[barras["grupo"] == grupo, "%"].to_numpy()
    fig.add_trace(go.Scatter(x=bordes, y=np.append(porcentaje, porcentaje[-1]), mode="lines", line_shape="hv",
                             line={"color": color, "width": 2.5}, name=grupo))
fig.update_layout(title="Minutos jugados al reseñar, por pulgar (escala log)", legend_title="pulgar")
marcas = [10, 60, 120, 600, 6_000, 60_000]
fig.update_xaxes(tickvals=np.log10(np.array(marcas) + 1), ticktext=[f"{m:,}" for m in marcas], title="minutos")
fig.add_vline(x=np.log10(ex.UMBRAL_REEMBOLSO + 1), line_dash="dash", line_color=ex.PALETA["GRIS"],
              annotation_text="120 min: reembolso", annotation_position="top left")
fig.add_vline(x=np.log10(181), line_dash="dot", line_color=ex.PALETA["GRIS"],
              annotation_text="180 min: salto de positivas", annotation_position="top right")
fig.update_yaxes(title="% de las reseñas del grupo")
fig.show()

In [47]:
sensibilidad = ex.sensibilidad_umbral(resenas_v1)
display(sensibilidad)
assert sensibilidad["Spearman con 120"].min() > 0.9

,Y=1,prevalencia,Spearman con 120
umbral (min),,,
60,1649,0.0133,0.964
90,2250,0.0181,0.989
120,2714,0.0219,1.000
180,3349,0.0270,0.993


**Qué vemos.** La señal es rara: un modelo que siempre dice «no» acierta casi siempre y no encuentra ni un
caso, así que la exactitud no sirve. El PR-AUC de ese modelo trivial es la prevalencia: es el piso contra el
que se compara. Las negativas llegan antes que las positivas: una parte varias veces mayor se escribe antes de
los 120 minutos, aunque la mayoría de las negativas llega con muchas horas encima.

En 120 minutos no hay escalón: las reseñas por minuto justo antes y justo después son casi iguales en las dos
curvas (las cifras, en la celda de la tabla). El salto que se ve en la figura está en **180 minutos** y solo en
las positivas: de un minuto a otro se multiplican varias veces, en decenas de juegos, mientras las negativas
cambian poco. Su causa no está en estos datos, y no toca la señal, que se define en 120 minutos y con reseñas
negativas. Con
umbrales de 60, 90 o 180 minutos cambia cuántas reseñas cuentan, pero el orden de los juegos casi no cambia.

**Qué decidimos.** El umbral se queda en **120 minutos**, la ventana de reembolso de Steam: no se elige con
datos, y los datos muestran que moverlo no cambia qué juegos tienen más señal. La métrica es **PR-AUC**.

## 3.4 Variables del juego

**Pregunta:** ¿qué características del juego acompañan una tasa de señal más alta?

Se mira por juego (83 puntos), no por reseña: las variables no cambian dentro de un juego, y 124 mil filas
repetidas darían una falsa sensación de precisión.

In [48]:
por_juego = por_juego.assign(
    precio_pesos=por_juego["precio_final"] / 100,
    imputado=ex.precio_imputado(por_juego),
    nota=np.where(por_juego["metacritic"].notna(), "con nota", "sin nota"),
)
de_pago = por_juego[por_juego["es_gratis"] == 0]
fig = px.scatter(de_pago[~de_pago["imputado"]], x="precio_pesos", y="tasa", hover_name="nombre", log_x=True,
                 title="Tasa de señal contra precio (juegos de pago)")
fig.update_traces(marker={"color": ex.PALETA["AZUL"], "size": 8}, name="precio conocido", showlegend=True)
imputados = de_pago[de_pago["imputado"]]
fig.add_trace(go.Scatter(x=[de_pago["precio_pesos"].min() / 3] * len(imputados), y=imputados["tasa"], mode="markers",
                         hovertext=imputados["nombre"], marker={"color": ex.PALETA["ROJO"], "symbol": "x", "size": 12},
                         name="precio imputado como 0"))
fig.add_annotation(text="✖ precio imputado como 0: " + ", ".join(imputados["nombre"]), xref="paper", yref="paper",
                   x=0, y=1.04, showarrow=False, xanchor="left", font={"color": ex.PALETA["ROJO"], "size": 12})
fig.update_layout(xaxis_title="precio en pesos (escala log)", yaxis_title="tasa de señal", yaxis_tickformat=".0%")
fig.show()

In [49]:
# Solo puntos: con tan pocos juegos gratis, una caja aparentaría una distribución que no hay.
fig = px.strip(por_juego.assign(tipo=np.where(por_juego["es_gratis"] == 1, "gratis", "de pago")), x="tipo", y="tasa",
               hover_name="nombre", title="Tasa de señal: gratis contra de pago")
fig.update_traces(marker={"color": ex.PALETA["AZUL"], "size": 8})
fig.update_layout(yaxis_tickformat=".0%", xaxis_title="", yaxis_title="tasa de señal")
fig.show()

In [50]:
fig = px.scatter(por_juego, x="descuento", y="tasa", hover_name="nombre", title="Tasa de señal contra descuento")
fig.update_traces(marker={"color": ex.PALETA["AZUL"], "size": 8})
fig.update_layout(xaxis_title="descuento (%)", yaxis_title="tasa de señal", yaxis_tickformat=".0%")
fig.show()

In [51]:
fig = px.strip(por_juego.assign(metacritic=por_juego["metacritic"].fillna(-1)), x="metacritic", y="tasa", color="nota",
               hover_name="nombre", title="Tasa de señal contra nota de la crítica (los sin nota, aparte)",
               color_discrete_map={"con nota": ex.PALETA["AZUL"], "sin nota": ex.PALETA["ROJO"]})
fig.update_layout(xaxis_title="Metacritic (−1 = sin nota)", yaxis_title="tasa de señal", yaxis_tickformat=".0%")
fig.show()

In [52]:
correlaciones = ex.correlaciones_del_juego(por_juego)
display(correlaciones)
conteos_juego = pd.Series({"gratis": int(por_juego["es_gratis"].sum()), "de pago sin precio": int(por_juego["imputado"].sum()),
                           "sin nota de la crítica": int(por_juego["metacritic"].isna().sum())}, name="juegos")
display(conteos_juego.to_frame())
# Lo único claro es la crítica: sin nota, o con nota más baja, hay más señal.
assert correlaciones.loc["tiene nota de la crítica", "Spearman"] < 0 and correlaciones.loc["tiene nota de la crítica", "p"] < 0.01
assert correlaciones.loc["nota de la crítica (con nota)", "Spearman"] < 0 and correlaciones.loc["nota de la crítica (con nota)", "p"] < 0.01
assert correlaciones.loc["precio (juegos de pago)", "p"] > 0.05

,juegos,Spearman,p
variable,,,
precio (juegos de pago),79,0.177,0.1182
es gratis,83,0.177,0.1089
descuento,83,-0.038,0.7312
tiene nota de la crítica,83,-0.402,0.0002
nota de la crítica (con nota),60,-0.422,0.0008


,juegos
gratis,2
de pago sin precio,2
sin nota de la crítica,23


### Revisión multivariada

Una por una, las variables pueden engañar: el precio podría parecer importante solo porque los juegos caros
tienden a no tener nota, o al revés. Aquí van juntas: los coeficientes del modelo B+ (variables
estandarizadas) con su intervalo al 95 % por **bootstrap sobre juegos**, 1,000 réplicas que remuestrean
appids, porque el juego es la unidad que varía.

In [53]:
coeficientes = ex.coeficientes_con_bootstrap(df_v1)
display(coeficientes.round(3))
assert (coeficientes["coeficiente"] - coeficientes["fila por fila"]).abs().max() < 1e-8, "el atajo por juego no reproduce el ajuste"
# Lo que sostiene la decisión de abajo: con todo junto, solo la crítica se distingue de cero.
assert not coeficientes.loc[["metacritic_disponible", "metacritic"], "cruza el cero"].any()
assert coeficientes.loc[["log_precio_final", "descuento", "es_gratis"], "cruza el cero"].all()

,coeficiente,fila por fila,IC 2.5 %,IC 97.5 %,réplicas > 0,cruza el cero
variable,,,,,,
es_gratis,0.078,0.078,-0.119,0.787,0.681,True
log_precio_final,0.258,0.258,-0.012,0.944,0.966,True
descuento,-0.143,-0.143,-0.310,0.071,0.092,True
metacritic_disponible,-0.825,-0.825,-1.043,-0.555,0.000,False
metacritic,-0.340,-0.340,-0.561,-0.072,0.000,False


In [54]:
orden_coef = coeficientes.sort_values("coeficiente")
fig = go.Figure(go.Scatter(
    x=orden_coef["coeficiente"], y=orden_coef.index, mode="markers",
    marker={"color": np.where(orden_coef["cruza el cero"], ex.PALETA["GRIS"], ex.PALETA["AZUL"]), "size": 11},
    error_x={"type": "data", "symmetric": False, "array": orden_coef["IC 97.5 %"] - orden_coef["coeficiente"],
             "arrayminus": orden_coef["coeficiente"] - orden_coef["IC 2.5 %"], "color": ex.PALETA["GRIS"]},
))
fig.add_vline(x=0, line_dash="dot", line_color=ex.PALETA["ROJO"])
fig.update_layout(title="Coeficientes del modelo B+ con IC al 95 % (bootstrap sobre juegos; gris = cruza el cero)",
                  xaxis_title="coeficiente (variable estandarizada; > 0 sube el riesgo estimado)", yaxis_title="")
fig.show()

**Qué vemos.** La asociación clara es con la crítica: los juegos sin nota, y los que tienen una nota más
baja, tienen más señal. Con el precio, la gratuidad y el descuento la relación es débil y no se distingue del
azar con 83 juegos; además, en data-v1 casi no hay juegos gratis, así que esa comparación dice poco. Los dos
juegos con precio imputado (cruces rojas) quedan fuera de la escala de precio: el modelo los lee como si
costaran 0.

Con todas juntas, solo la crítica tiene un efecto que no cruza el cero: no tener nota y tener una nota más
baja suben el riesgo estimado. El precio empuja hacia arriba en casi todas las réplicas, pero su intervalo
cruza el cero; el descuento y la gratuidad, también.

**Qué decidimos.**

- **El modelo no se toca:** sigue con las cinco variables del juego (gratuidad, precio en log, descuento,
  si tiene nota y la nota), que se conocen antes de comprar y valen igual para cualquier persona. La falta
  de nota entra como dato (`metacritic_disponible`), no se rellena en silencio. El precio imputado queda
  documentado como limitación.
- **Con la crítica controlada, el precio no se distingue de cero**, y tampoco el descuento ni la
  gratuidad. Eso cambia **cómo se explica** el riesgo (decisión del equipo, que se implementa en la ronda
  «explicar el riesgo»):
  - el orden de los factores de «Qué mueve esta estimación» sigue el aporte real que calculó el modelo
    para cada juego;
  - cada factor lleva su nivel de evidencia: sólida para la crítica; débil para precio, descuento y
    gratuidad («con 83 juegos no se distingue de cero»);
  - los juegos gratis llevan una nota de extrapolación, porque en data-v1 casi no había juegos gratis
    (la cifra está en la tabla de conteos de arriba);
  - Nia sigue la misma regla.
- La tabla de coeficientes con sus intervalos es la evidencia. La regla está en la
  [ficha de la mejora](https://github.com/fernandoaxelramirezgomez-coder/nexplay/blob/master/docs/historial/mejoras/01-antes-del-reembolso.md) y la comprueba
  [`backend/calidad/verificar_factores.py`](https://github.com/fernandoaxelramirezgomez-coder/nexplay/blob/master/backend/calidad/verificar_factores.py).

## 3.5 Variables del jugador

**Pregunta:** ¿el perfil del jugador agrega algo al riesgo del juego?

In [55]:
privados = (resenas_v1["num_games_owned"] == 0).mean()
print(f"reseñas con perfil privado (num_games_owned == 0): {privados:.1%}")
jugador = ex.juego_contra_compra(df_v1)
display(pd.DataFrame({"PR-AUC media": [jugador["juego"].mean(), jugador["compra"].mean()],
                      "std entre folds": [jugador["juego"].std(), jugador["compra"].std()]},
                     index=["juego (producción)", "compra (+ compras del jugador)"]).round(4))
print(f"aporte del lado del jugador: {jugador['aporte']:.1%} del PR-AUC; "
      f"diferencia de medias {jugador['diferencia']:.4f} contra std entre folds {jugador['std_juego']:.4f}")
# Las mismas condiciones que sostienen la decisión en 01: si otro corte las rompiera, el notebook falla aquí.
assert jugador["aporte"] < 0.05, "el lado del jugador ya no es marginal"
assert abs(jugador["diferencia"]) < jugador["std_juego"], "la diferencia ya no cabe en el ruido entre folds"

reseñas con perfil privado (num_games_owned == 0): 59.9%


,PR-AUC media,std entre folds
juego (producción),0.0694,0.0415
compra (+ compras del jugador),0.0709,0.0270


aporte del lado del jugador: 2.1% del PR-AUC; diferencia de medias 0.0015 contra std entre folds 0.0415


| Variable | Cuándo se conoce | Por qué no entra |
|---|---|---|
| `num_games_owned` | en la reseña; en la app se declara «compras al año» | aporta dentro del ruido entre folds; además, 0 es un perfil privado, no una biblioteca vacía |
| `num_reviews` | cambia con el tiempo; se lee al descargar | fuga temporal: describe al autor después de reseñar |
| `steam_purchase`, `received_for_free` | atributos de esa reseña | fuga: solo existen porque la reseña ya se escribió |
| `written_during_early_access` | atributo de esa reseña | fuga, y no se sabe antes de comprar |

**Qué vemos.** Agregar las compras declaradas del jugador mueve el PR-AUC mucho menos que la variación normal
entre folds. Las demás variables del jugador solo existen porque la reseña ya se escribió.

**Qué decidimos.** El riesgo es **del juego**, igual para cualquier persona (modelo B+). El perfil declarado
sirve para la afinidad, no para mover el riesgo. El detalle está en `01_modelo_riesgo.ipynb`, secciones 6 y 7, y en
`docs/evidencia/simulacion_123.txt` (30 particiones distintas).

## 3.6 Los 83 contra los 40

**Pregunta:** ¿los 40 juegos que llegaron después se parecen a los 83 con los que se entrenó? Esta sección
solo compara: **no se decide nada aquí**.

In [56]:
por_juego_ext = ex.tasa_por_juego(resenas_ext, juegos_ext)
comparacion = pd.concat([por_juego.assign(release="data-v1 (83)"), por_juego_ext.assign(release="externos (40)")])
comparacion = comparacion.assign(precio_pesos=comparacion["precio_final"] / 100)
for columna, titulo in (("precio_pesos", "precio en pesos"), ("metacritic", "nota de la crítica"), ("reseñas", "reseñas por juego")):
    fig = px.histogram(comparacion, x=columna, color="release", barmode="overlay", histnorm="percent", opacity=0.6,
                       nbins=20, color_discrete_map=ex.COLOR_RELEASE, title=f"{titulo.capitalize()}: 83 contra 40")
    fig.update_layout(xaxis_title=titulo, yaxis_title="% de los juegos")
    fig.show()

In [57]:
resumen_releases = comparacion.groupby("release").agg(
    juegos=("tasa", "size"), gratis=("es_gratis", "sum"), sin_nota=("metacritic", lambda s: s.isna().sum()),
    precio_mediano=("precio_pesos", "median"), tasa_mediana=("tasa", "median"),
)
display(resumen_releases.round(3))

,juegos,gratis,sin_nota,precio_mediano,tasa_mediana
release,,,,,
data-v1 (83),83,2,23,400.00,0.008
externos (40),40,5,10,269.99,0.014


In [58]:
# Las mismas validaciones y la misma limpieza, aplicadas a los 40.
display(pd.concat({"data-v1": ex.llaves(juegos_v1, resenas_v1)["casos"],
                   "externos": ex.llaves(juegos_ext, resenas_ext)["casos"]}, axis=1))
display(pd.concat({"data-v1": ex.resumen_de_rango(casos)["casos"],
                   "externos": ex.resumen_de_rango(ex.casos_de_rango(juegos_ext, resenas_ext))["casos"]}, axis=1))
limpio_ext, pasos_ext = li.limpiar(resenas_ext)
display(li.bitacora(pasos_ext))

,data-v1,externos
chequeo,,
recommendationid repetido,0,0
appid de reseñas que no está en juegos,0,0
juegos sin reseñas,0,0
appid repetido en juegos,0,0


,data-v1,externos
regla,,
playtime_at_review ≥ 0,0,0
"voted_up ∈ {0, 1}",0,0
creada ≤ editada ≤ descargada,0,0
precio ≥ 0,0,0
descuento entre 0 y 100,0,0
Metacritic entre 0 y 100 o nulo,0,0
playtime_at_review ≤ playtime_forever,1,0
es_gratis coherente con el precio,2,0


,regla,filas antes,filas después,afectadas,motivo
0,quitar_duplicados_exactos,60395,60360,35,texto idéntico de 8+ palabras; queda la más antigua
1,quitar_vacias,60360,60101,259,sin texto no hay nada que leer
2,marcar_cortas,60101,60101,13424,"menos de 3 palabras; se marcan, no se quitan"
3,marcar_no_ingles,60101,60101,759,"otro idioma con confianza (lingua); se marcan, no se quitan"
4,marcar_plantillas,60101,60101,95,"reseña plantilla («☐ Bad ☑ Good»); se marcan, no se quitan"
5,normalizar_texto,60101,60101,46013,"minúsculas, sin BBCode ni URLs, negaciones como palabra; agrega texto_norm"


**Qué vemos.** Los 40 externos pasan las mismas validaciones, y la limpieza los afecta en proporciones
parecidas, salvo los duplicados, que son muchos menos. Sus distribuciones de precio, nota y número de reseñas
se solapan con las de los 83, pero la mezcla cambia: hay más juegos gratis, son más baratos y su tasa de
señal mediana es más alta. Por eso la prueba externa es más dura que la validación cruzada.

**Qué decidimos.** Nada: los 40 son la prueba externa del modelo y de los modelos de texto, y no se usan para
elegir variables ni umbrales.

## 3.7 Texto

**Pregunta:** ¿qué distingue el texto de una negativa temprana? Todo sobre el conjunto limpio.

In [59]:
grupo = ex.grupo_de_resena(limpio)
balance = limpio["voted_up"].map({1: "positiva", 0: "negativa"}).value_counts()
display(pd.DataFrame({"reseñas": balance, "%": (100 * balance / balance.sum()).round(1)}))
display(grupo.value_counts().rename("reseñas").to_frame())

,reseñas,%
voted_up,,
positiva,105885,86.1
negativa,17059,13.9


,reseñas
positiva,105885
negativa tardía,14353
negativa temprana,2706


In [60]:
largo = ex.cuantiles_por_grupo(li.palabras(limpio["texto"]), grupo)
display(largo)
assert largo.loc["negativa temprana", "mediana"] < largo.loc["negativa tardía", "mediana"]
fig = go.Figure([
    go.Box(name=g, x=[g], q1=[f["q1"]], median=[f["mediana"]], q3=[f["q3"]], lowerfence=[f["p5"]], upperfence=[f["p95"]],
           marker_color=ex.COLOR_GRUPO[g], line_color=ex.COLOR_GRUPO[g])
    for g, f in largo.iterrows()
])
fig.update_layout(title="Palabras por reseña, por grupo (bigotes: percentiles 5 y 95)", yaxis_type="log",
                  yaxis_title="palabras", showlegend=False)
fig.show()

,p5,q1,mediana,q3,p95
negativa tardía,1.0,9.0,27.0,78.0,300.00
negativa temprana,2.0,8.0,22.0,53.0,165.75
positiva,1.0,3.0,8.0,24.0,129.00


In [61]:
def barras_de_log_odds(tabla, titulo, lado_a, lado_b, n=12):
    extremos = pd.concat([tabla.head(n).assign(lado=lado_a), tabla.tail(n).assign(lado=lado_b)]).sort_values("z")
    fig = px.bar(extremos, x="z", y="palabra", color="lado", orientation="h", title=titulo,
                 color_discrete_map={lado_a: ex.COLOR_GRUPO["negativa temprana"], lado_b: ex.COLOR_GRUPO.get(lado_b, ex.PALETA["GRIS"])})
    fig.update_layout(yaxis_title="", xaxis_title="z del log-odds (más a la derecha: más de las negativas tempranas)")
    fig.update_yaxes(dtick=1, tickfont={"size": 11})
    fig.show()


tempranas = limpio.loc[grupo == "negativa temprana", "texto_norm"]
contra_positivas = ex.log_odds_informativo(tempranas, limpio.loc[grupo == "positiva", "texto_norm"], li.STOPWORDS_SIN_NEGACIONES)
barras_de_log_odds(contra_positivas, "Palabras que distinguen a las negativas tempranas de las positivas", "negativa temprana", "positiva")
assert "not" in contra_positivas["palabra"].head(15).tolist(), "la negación ya no está entre las palabras que más distinguen"

In [62]:
contra_tardias = ex.log_odds_informativo(tempranas, limpio.loc[grupo == "negativa tardía", "texto_norm"], li.STOPWORDS_SIN_NEGACIONES)
barras_de_log_odds(contra_tardias, "Palabras que distinguen a las negativas tempranas de las tardías", "negativa temprana", "negativa tardía")
assert {"tutorial", "refund", "account"} & set(contra_tardias["palabra"].head(15)), "cambiaron las palabras de las primeras horas"

In [63]:
refund = ex.menciones(limpio["texto"], grupo, r"\brefund")
display((100 * refund).round(1).rename("% que menciona «refund»").to_frame())
assert refund["negativa temprana"] > refund["negativa tardía"] > refund["positiva"]

,% que menciona «refund»
negativa tardía,2.9
negativa temprana,9.5
positiva,0.2


In [64]:
peso_plantillas = limpio.groupby(grupo)["es_plantilla"].agg(["sum", "mean"]).rename(columns={"sum": "plantillas", "mean": "proporción"})
display(peso_plantillas.round(4))

,plantillas,proporción
negativa tardía,5,0.0003
negativa temprana,3,0.0011
positiva,183,0.0017


**Qué vemos.** Las positivas son la gran mayoría. Las negativas tempranas son más cortas que las tardías y
hablan de lo que pasa en las primeras horas: que no arranca, la cuenta o el launcher, el tutorial, el
reembolso. «not» aparece entre las palabras que más las distinguen: por eso la normalización conserva las
negaciones. Las tardías hablan de horas, contenido y jefes.

«refund» aparece mucho más en las negativas tempranas. Es una **señal legítima que escribe el autor**, no
una fuga: está en el texto que la persona publicó, y el texto nunca entra al modelo de riesgo (su `Y` sale de
esas mismas reseñas). Las plantillas pesan poco en todos los grupos.

**Qué decidimos.** Los modelos de texto trabajan sobre `texto_norm`, con las negaciones como palabra.
«refund» no se quita del texto.

## 3.8 Motivos por palabras clave

**Pregunta:** ¿cuántas negativas tempranas explica hoy la lista de palabras clave de `/explicacion`? Es la
línea base de la Parte B.

In [65]:
senal_limpia = limpio[ex.senal(limpio)]
tabla_motivos = motivos.tabla_de_motivos(senal_limpia["texto"])
cobertura_motivos = tabla_motivos.any(axis=1).mean()
print(f"COBERTURA: {cobertura_motivos:.1%} de las {len(senal_limpia):,} negativas tempranas tienen al menos un motivo; "
      f"{1 - cobertura_motivos:.1%} quedan sin motivo.")
print(f"con dos motivos o más: {(tabla_motivos.sum(axis=1) >= 2).mean():.1%}")

COBERTURA: 34.0% de las 2,706 negativas tempranas tienen al menos un motivo; 66.0% quedan sin motivo.
con dos motivos o más: 9.2%


In [66]:
distribucion = tabla_motivos.sum().sort_values()
fig = px.bar(x=distribucion.values, y=distribucion.index, orientation="h", title="Negativas tempranas por motivo",
             labels={"x": "reseñas", "y": ""})
fig.update_traces(marker_color=ex.PALETA["AZUL"])
fig.show()

In [67]:
fig = px.imshow(ex.coocurrencia(tabla_motivos), text_auto=True, title="Co-ocurrencia de motivos (la diagonal es el total)",
                color_continuous_scale=[[0, "#ffffff"], [1, ex.PALETA["AZUL"]]])
fig.show()

**Qué vemos.** Las palabras clave dejan **sin motivo a la mayoría** de las negativas tempranas: la cobertura
de la celda de arriba es la cifra central de esta sección. Rendimiento y bugs dominan, y los motivos se
combinan poco entre sí.

**Qué decidimos.** Esa cobertura es la **justificación de la Parte B**: probar si un modelo que lee el texto
completo asigna motivos a más reseñas, con un conjunto de referencia etiquetado a mano y la regla de decisión
fijada antes de mirar.

## 3.9 Opcional: proyección de embeddings

**Intuición visual, sin conclusiones propias.** 2,000 reseñas (la misma cantidad de cada grupo), en inglés y
no cortas, pasadas por `all-MiniLM-L6-v2` → PCA a 50 dimensiones → t-SNE a 2. Las distancias de t-SNE no se
interpretan; la figura solo sugiere si el texto tiene estructura.

In [68]:
if CORRER_EMBEDDINGS:
    legibles = limpio[~limpio["es_corta"] & ~limpio["no_ingles"]]
    muestra = ex.muestra_estratificada(legibles, grupo.loc[legibles.index], 2_000)
    xy = ex.proyeccion_2d(muestra["texto_norm"], MODELO_EMBEDDINGS, REVISION_EMBEDDINGS)
    proyeccion = pd.DataFrame({"x": xy[:, 0], "y": xy[:, 1], "pulgar": muestra["voted_up"].map({1: "positiva", 0: "negativa"}).to_numpy()})
    fig = px.scatter(proyeccion, x="x", y="y", color="pulgar", opacity=0.6, title="Proyección 2D de 2,000 reseñas (intuición visual)",
                     color_discrete_map={"positiva": ex.PALETA["VERDE_OSC"], "negativa": ex.PALETA["ROJO"]})
    fig.update_xaxes(showticklabels=False, title="")
    fig.update_yaxes(showticklabels=False, title="")
    fig.show()
else:
    print("CORRER_EMBEDDINGS = False: sección saltada.")

# Resumen de decisiones

Cada decisión, la sección que la justifica y la cifra que la respalda, calculada arriba.

In [69]:
con_senal = ex.senal(limpio)
resumen = pd.DataFrame([
    ("Validar con GroupKFold por appid", "3.2",
     f"la tasa varía {dispersion:.0f}× más de lo esperado; KFold aleatorio {validaciones.iloc[0, 0]:.3f} contra GroupKFold {validaciones.iloc[1, 0]:.3f}"),
    ("Partición congelada por firma", "1.6, Bloque 2 y 3.2",
     f"firma {chequeos.loc['firma', 'valor'][:12]}…; 20 particiones distintas dan de {particiones.min():.3f} a {particiones.max():.3f} "
     f"(media {particiones.mean():.3f}); con la congelada, quitar duplicados deja {duplicados['PR-AUC después (partición congelada)']:.4f} "
     f"contra {duplicados['PR-AUC antes']:.4f}"),
    ("Umbral de 120 min (reembolso), sin ajustar", "3.3",
     f"Spearman con 60/90/180 min: {sensibilidad['Spearman con 120'].min():.2f} a {sensibilidad.drop(120)['Spearman con 120'].max():.2f}"),
    ("PR-AUC, no exactitud", "3.3",
     f"«siempre 0» acierta {exactitud.iloc[0, 0]:.1%} con PR-AUC {exactitud.iloc[0, 1]:.3f} (= prevalencia {prevalencia:.3f})"),
    ("Orden por aporte real; evidencia sólida para la crítica y débil para precio, descuento y gratuidad", "3.4",
     "; ".join(f"{v} {c['coeficiente']:+.2f} [{c['IC 2.5 %']:+.2f}, {c['IC 97.5 %']:+.2f}]" for v, c in coeficientes.iterrows())),
    ("Riesgo del juego, sin variables del jugador (B+)", "3.5",
     f"aporte del jugador {jugador['aporte']:.1%} contra std entre folds {jugador['std_juego']:.4f}"),
    ("Precio imputado como 0, sin corregir en el modelo; la ficha lo mostrará con el aviso «Estimación menos confiable»", "1.4",
     ", ".join(casos["es_gratis coherente con el precio"]["nombre"])),
    ("Quitar duplicados de 8+ palabras (solo texto)", "1.5 y 2.1",
     f"{len(pasos[0].afectadas):,} filas; {copias_cruzadas} de {largas['textos en 2+ juegos']} copias entre juegos en folds distintos"),
    ("Marcar cortas, no quitarlas", "2.3",
     f"{limpio['es_corta'].mean():.1%} de las reseñas y {(limpio['es_corta'] & con_senal).sum() / con_senal.sum():.1%} de la señal; "
     f"quitarlas movería el PR-AUC a {impacto.loc['quitar cortas (hipótesis)', 'PR-AUC después (partición congelada)']:.3f}"),
    ("Embeddings en inglés (all-MiniLM-L6-v2)", "2.4",
     f"{en_ingles:.1%} en inglés o sin idioma determinado; {negativas_otro_idioma} negativas en otro idioma"),
    ("Limpieza no aplicada al modelo congelado", "Bloque 2",
     f"con la partición congelada, duplicados y vacías dejan el PR-AUC en {impacto.loc['quitar duplicados', 'PR-AUC después (partición congelada)']:.4f} "
     f"y {impacto.loc['quitar vacías', 'PR-AUC después (partición congelada)']:.4f} (antes {impacto.loc['quitar duplicados', 'PR-AUC antes']:.4f})"),
    ("«refund» es señal legítima, no fuga", "3.7",
     f"lo menciona {refund['negativa temprana']:.1%} de las tempranas, {refund['negativa tardía']:.1%} de las tardías y {refund['positiva']:.1%} de las positivas"),
    ("Hace falta la Parte B (motivos)", "3.8",
     f"las palabras clave cubren {cobertura_motivos:.1%} de las negativas tempranas"),
], columns=["decisión", "sección", "cifra"])
display(resumen.set_index("decisión"))

,sección,cifra
decisión,,
Validar con GroupKFold por appid,3.2,la tasa varía 107× más de lo esperado; KFold aleatorio 0.080 contra GroupKFold 0.069
Partición congelada por firma,"1.6, Bloque 2 y 3.2","firma 416aa8243b57…; 20 particiones distintas dan de 0.048 a 0.087 (media 0.069); con la congelada, quitar duplicados deja 0.0695 contra 0.0694"
"Umbral de 120 min (reembolso), sin ajustar",3.3,Spearman con 60/90/180 min: 0.96 a 0.99
"PR-AUC, no exactitud",3.3,«siempre 0» acierta 97.8% con PR-AUC 0.022 (= prevalencia 0.022)
"Orden por aporte real; evidencia sólida para la crítica y débil para precio, descuento y gratuidad",3.4,"es_gratis +0.08 [-0.12, +0.79]; log_precio_final +0.26 [-0.01, +0.94]; descuento -0.14 [-0.31, +0.07]; metacritic_disponible -0.82 [-1.04, -0.55]; metacritic -0.34 [-0.56, -0.07]"
"Riesgo del juego, sin variables del jugador (B+)",3.5,aporte del jugador 2.1% contra std entre folds 0.0415
"Precio imputado como 0, sin corregir en el modelo; la ficha lo mostrará con el aviso «Estimación menos confiable»",1.4,"Grand Theft Auto V Legacy, New World: Aeternum"
Quitar duplicados de 8+ palabras (solo texto),1.5 y 2.1,483 filas; 52 de 57 copias entre juegos en folds distintos
"Marcar cortas, no quitarlas",2.3,22.7% de las reseñas y 8.7% de la señal; quitarlas movería el PR-AUC a 0.080
